# Project 67 — Local Structured Output Reliability Test
## Test JSON/Pydantic Schema Adherence Across Complexities

**Stack:** LangChain · Ollama · Pydantic · pandas · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pydantic pandas

## Step 1 — Define Test Schemas (Easy → Hard)

In [ ]:
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field
import json, time, pandas as pd

llm = ChatOllama(model="qwen3:8b", temperature=0.0)

# Level 1: Simple flat schema
class PersonInfo(BaseModel):
    name: str
    age: int
    city: str
    occupation: str

# Level 2: Nested schema
class Address(BaseModel):
    street: str
    city: str
    zip_code: str
    country: str

class ContactInfo(BaseModel):
    person: PersonInfo
    address: Address
    phone: str
    email: str

# Level 3: Lists and enums
class SkillEntry(BaseModel):
    name: str
    level: str = Field(description="beginner, intermediate, expert")
    years: int

class Resume(BaseModel):
    name: str
    title: str
    skills: list[SkillEntry]
    education: list[str]
    summary: str

# Level 4: Complex nested with constraints
class APIEndpoint(BaseModel):
    method: str = Field(description="GET, POST, PUT, DELETE")
    path: str
    description: str
    parameters: list[str]
    response_codes: list[int]
    requires_auth: bool

class APISpec(BaseModel):
    service_name: str
    version: str
    base_url: str
    endpoints: list[APIEndpoint]

test_schemas = [
    ("L1-flat", PersonInfo, "Extract: Alice Johnson, 32, from Seattle, software engineer"),
    ("L2-nested", ContactInfo, "Extract: Bob Smith, 45, NYC, teacher. "
     "Address: 123 Main St, New York, 10001, USA. Phone: 555-1234, bob@email.com"),
    ("L3-lists", Resume, "Create a resume for Carol, Senior Data Scientist, "
     "skills: Python (expert, 8yr), SQL (intermediate, 4yr), ML (expert, 6yr). "
     "Education: MIT CS PhD, Stanford BS. Summary: Experienced DS leader."),
    ("L4-complex", APISpec, "Design a user management API v2.0 at /api with endpoints: "
     "GET /users (list, params: page,limit, codes: 200,401), "
     "POST /users (create, params: name,email, codes: 201,400,401, auth required), "
     "DELETE /users/:id (delete, codes: 204,404,401, auth required)"),
]
print(f"Test schemas: {len(test_schemas)} levels of complexity")

## Step 2 — Run Structure Tests

In [ ]:
results = []
for level, schema, prompt in test_schemas:
    structured_llm = llm.with_structured_output(schema)
    start = time.time()
    try:
        output = structured_llm.invoke(prompt)
        valid = True
        output_dict = output.model_dump()
        error = ""
    except Exception as e:
        valid = False
        output_dict = {}
        error = str(e)[:100]
    elapsed = time.time() - start

    results.append({
        "level": level, "valid": valid, "latency": round(elapsed, 2),
        "output_size": len(json.dumps(output_dict, default=str)),
        "error": error,
    })
    icon = "✓" if valid else "✗"
    print(f"  {icon} {level}: {'PASS' if valid else 'FAIL'} ({elapsed:.1f}s)")
    if valid:
        print(f"    {json.dumps(output_dict, indent=2, default=str)[:250]}")
    else:
        print(f"    Error: {error}")
    print()

## Step 3 — Robustness: Multiple Runs

In [ ]:
# Run each schema 3 times to check consistency
consistency_results = []
for level, schema, prompt in test_schemas:
    structured_llm = llm.with_structured_output(schema)
    passes = 0
    for trial in range(3):
        try:
            output = structured_llm.invoke(prompt)
            passes += 1
        except Exception:
            pass
    consistency_results.append({
        "level": level,
        "pass_rate": f"{passes}/3",
        "consistent": passes == 3,
    })
    print(f"  {level}: {passes}/3 passes {'✓' if passes == 3 else '⚠'}")

print(f"\nFully consistent schemas: {sum(1 for r in consistency_results if r['consistent'])}/{len(consistency_results)}")

## Step 4 — Results Dashboard

In [ ]:
rdf = pd.DataFrame(results)
print("STRUCTURED OUTPUT BENCHMARK")
print("=" * 60)
print(rdf[["level","valid","latency","output_size","error"]].to_string(index=False))

pass_rate = rdf["valid"].mean()
print(f"\nOverall pass rate: {pass_rate:.0%}")
print(f"Avg latency: {rdf['latency'].mean():.2f}s")
print(f"Fastest: {rdf['latency'].min():.2f}s ({rdf.loc[rdf['latency'].idxmin(), 'level']})")
print(f"Slowest: {rdf['latency'].max():.2f}s ({rdf.loc[rdf['latency'].idxmax(), 'level']})")

## What You Learned
- **Schema complexity tiers** from flat to deeply nested
- **Structured output reliability** testing
- **Consistency checks** with repeated runs
- **Performance benchmarking** across schema types